# Generate LaTeX Results Table

This notebook reads experiment results from `experiments/QNN_integration/results`, computes mean and standard deviation for the test metrics, and generates a LaTeX table following `experiments/QNN_integration/analysis/latex_table_template.tex`.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "experiments" / "QNN_integration" / "results").exists():
            return candidate
    raise FileNotFoundError("Could not locate experiments/QNN_integration/results from the current working directory.")


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "experiments" / "QNN_integration" / "results"
ANALYSIS_DIR = PROJECT_ROOT / "experiments" / "QNN_integration" / "analysis"
TEMPLATE_PATH = ANALYSIS_DIR / "latex_table_template.tex"
OUTPUT_TEX = ANALYSIS_DIR / "latex_results_table.tex"

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Template:     {TEMPLATE_PATH}")
print(f"Output tex:   {OUTPUT_TEX}")

Project root: c:\Users\micha\OneDrive - Politechnika Śląska\dokumenty\Doktorat\Python\geqie
Results dir:  c:\Users\micha\OneDrive - Politechnika Śląska\dokumenty\Doktorat\Python\geqie\experiments\QNN_integration\results
Template:     c:\Users\micha\OneDrive - Politechnika Śląska\dokumenty\Doktorat\Python\geqie\experiments\QNN_integration\analysis\latex_table_template.tex
Output tex:   c:\Users\micha\OneDrive - Politechnika Śląska\dokumenty\Doktorat\Python\geqie\experiments\QNN_integration\analysis\latex_results_table.tex


In [2]:
MODEL_CONFIG = [
    {"slug": "Baseline_A_Dense", "model_name": "Baseline A:", "encoding": ""},
    {"slug": "Baseline_B_CNN", "model_name": "Baseline B:", "encoding": ""},
    {"slug": "Baseline_C", "model_name": "Baseline C:", "encoding": ""},
    {"slug": "Baseline_D", "model_name": "Baseline D:", "encoding": ""},
    {"slug": "Experiment_A_FRQI", "model_name": "GEQIE-FRQI:", "encoding": ""},
    {"slug": "Experiment_B_NEQR", "model_name": "GEQIE-NEQR:", "encoding": ""},
]

METRICS = ["accuracy", "precision", "recall", "f1", "loss"]


def load_subset_metrics(results_dir: Path, model_config: list[dict]) -> tuple[pd.DataFrame, list[str]]:
    frames = []
    skipped_models = []

    for config in model_config:
        model_dir = results_dir / config["slug"]
        metric_files = sorted(model_dir.glob("*/subset_test_metrics.csv"))

        if not metric_files:
            skipped_models.append(config["slug"])
            continue

        for metrics_path in metric_files:
            df = pd.read_csv(metrics_path, sep=";")
            required_columns = {"subset", *METRICS}
            missing_columns = sorted(required_columns.difference(df.columns))
            if missing_columns:
                raise ValueError(f"{metrics_path} is missing columns: {missing_columns}")

            df = df.copy()
            df["model_slug"] = config["slug"]
            df["model_name"] = config["model_name"]
            df["encoding_preprocessing"] = config["encoding"]
            df["run"] = metrics_path.parent.name
            df["metrics_file"] = str(metrics_path.relative_to(PROJECT_ROOT))
            frames.append(df)

    if not frames:
        raise FileNotFoundError(f"No subset_test_metrics.csv files found under {results_dir}")

    metrics = pd.concat(frames, ignore_index=True)
    for column in ["subset", *METRICS]:
        metrics[column] = pd.to_numeric(metrics[column], errors="raise")

    return metrics, skipped_models


raw_metrics, skipped_models = load_subset_metrics(RESULTS_DIR, MODEL_CONFIG)
print(f"Loaded {len(raw_metrics)} subset metric rows from {raw_metrics['run'].nunique()} run folders.")
if skipped_models:
    print("Models without metric files:", ", ".join(skipped_models))

display(raw_metrics.head())

Loaded 45 subset metric rows from 8 run folders.


,subset,loss,accuracy,precision,recall,f1,model_slug,model_name,encoding_preprocessing,run,metrics_file
0,1,0.836259,0.822917,0.830239,0.822917,0.823384,Baseline_A_Dense,Baseline A:,,29-04-2026-13-20,experiments\QNN_integration\results\Baseline_A...
1,2,0.813757,0.829861,0.833785,0.829861,0.828506,Baseline_A_Dense,Baseline A:,,29-04-2026-13-20,experiments\QNN_integration\results\Baseline_A...
2,3,0.835871,0.847222,0.852115,0.847222,0.847323,Baseline_A_Dense,Baseline A:,,29-04-2026-13-20,experiments\QNN_integration\results\Baseline_A...
3,4,0.724560,0.840278,0.840302,0.840278,0.838751,Baseline_A_Dense,Baseline A:,,29-04-2026-13-20,experiments\QNN_integration\results\Baseline_A...
4,5,0.752013,0.864583,0.872017,0.864583,0.862402,Baseline_A_Dense,Baseline A:,,29-04-2026-13-20,experiments\QNN_integration\results\Baseline_A...


In [3]:
summary_rows = []

for config in MODEL_CONFIG:
    model_metrics = raw_metrics.loc[raw_metrics["model_slug"] == config["slug"]]
    if model_metrics.empty:
        continue

    row = {
        "model_slug": config["slug"],
        "model_name": config["model_name"],
        "encoding_preprocessing": config["encoding"],
        "runs": model_metrics["run"].nunique(),
        "subsets": len(model_metrics),
    }
    for metric in METRICS:
        row[f"{metric}_mean"] = model_metrics[metric].mean()
        row[f"{metric}_std"] = model_metrics[metric].std(ddof=1)
    summary_rows.append(row)

summary_stats = pd.DataFrame(summary_rows)
display(summary_stats)

,model_slug,model_name,encoding_preprocessing,runs,subsets,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,loss_mean,loss_std
0,Baseline_A_Dense,Baseline A:,,4,20,0.845486,0.018217,0.849261,0.018447,0.845486,0.018217,0.844456,0.017624,0.791735,0.047502
1,Baseline_B_CNN,Baseline B:,,1,5,0.852083,0.017807,0.861561,0.013812,0.852083,0.017807,0.851276,0.017672,0.487188,0.079802
2,Baseline_C,Baseline C:,,1,5,0.120833,0.034496,0.131287,0.024484,0.120833,0.034496,0.099187,0.031400,17.351821,0.905011
3,Baseline_D,Baseline D:,,1,5,0.502778,0.052967,0.560317,0.032603,0.502778,0.052967,0.484662,0.059292,10.140583,1.827762
4,Experiment_A_FRQI,GEQIE-FRQI:,,1,5,0.763194,0.038305,0.786813,0.031621,0.763194,0.038305,0.757401,0.039535,0.796358,0.115442
5,Experiment_B_NEQR,GEQIE-NEQR:,,1,5,0.765972,0.024626,0.785395,0.017240,0.765972,0.024626,0.762022,0.024592,0.817878,0.081090


In [4]:
DECIMALS = 3
AS_PERCENT = False


def format_mean_std(mean: float, std: float, decimals: int = DECIMALS, as_percent: bool = AS_PERCENT) -> str:
    scale = 100 if as_percent else 1
    if pd.isna(std):
        return f"{mean * scale:.{decimals}f}"
    return rf"{mean * scale:.{decimals}f} $\pm$ {std * scale:.{decimals}f}"


formatted_table = pd.DataFrame(
    {
        "Model name": summary_stats["model_name"],
        "Encoding / preprocessing": summary_stats["encoding_preprocessing"],
        "Accuracy": [format_mean_std(row.accuracy_mean, row.accuracy_std) for row in summary_stats.itertuples()],
        "Precision": [format_mean_std(row.precision_mean, row.precision_std) for row in summary_stats.itertuples()],
        "Recall": [format_mean_std(row.recall_mean, row.recall_std) for row in summary_stats.itertuples()],
        "F1": [format_mean_std(row.f1_mean, row.f1_std) for row in summary_stats.itertuples()],
        "Loss": [format_mean_std(row.loss_mean, row.loss_std) for row in summary_stats.itertuples()],
    }
)

display(formatted_table)

,Model name,Encoding / preprocessing,Accuracy,Precision,Recall,F1,Loss
0,Baseline A:,,0.845 $\pm$ 0.018,0.849 $\pm$ 0.018,0.845 $\pm$ 0.018,0.844 $\pm$ 0.018,0.792 $\pm$ 0.048
1,Baseline B:,,0.852 $\pm$ 0.018,0.862 $\pm$ 0.014,0.852 $\pm$ 0.018,0.851 $\pm$ 0.018,0.487 $\pm$ 0.080
2,Baseline C:,,0.121 $\pm$ 0.034,0.131 $\pm$ 0.024,0.121 $\pm$ 0.034,0.099 $\pm$ 0.031,17.352 $\pm$ 0.905
3,Baseline D:,,0.503 $\pm$ 0.053,0.560 $\pm$ 0.033,0.503 $\pm$ 0.053,0.485 $\pm$ 0.059,10.141 $\pm$ 1.828
4,GEQIE-FRQI:,,0.763 $\pm$ 0.038,0.787 $\pm$ 0.032,0.763 $\pm$ 0.038,0.757 $\pm$ 0.040,0.796 $\pm$ 0.115
5,GEQIE-NEQR:,,0.766 $\pm$ 0.025,0.785 $\pm$ 0.017,0.766 $\pm$ 0.025,0.762 $\pm$ 0.025,0.818 $\pm$ 0.081


In [5]:
def latex_escape_text(value: object) -> str:
    replacements = {
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
    }
    text = "" if pd.isna(value) else str(value)
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def generate_latex_table(
    table: pd.DataFrame,
    caption: str = "Classification results across cross-validation subsets.",
    label: str = "tab:qnn_integration_results",
) -> str:
    lines = [
        r"% TABLE 1: Basic statistics:",
        r"\begin{table}[h]",
        rf"\caption{{{latex_escape_text(caption)}}}\label{{{label}}}%",
        r"\begin{tabular}{@{}lllllll@{}}",
        r"\toprule",
        r"Model name & Encoding / preprocessing & Accuracy & Precision & Recall & F1 & Loss \\",
        r"\midrule",
    ]

    metric_columns = ["Accuracy", "Precision", "Recall", "F1", "Loss"]
    for _, row in table.iterrows():
        cells = [
            latex_escape_text(row["Model name"]),
            latex_escape_text(row["Encoding / preprocessing"]),
            *[str(row[column]) for column in metric_columns],
        ]
        lines.append(" & ".join(cells) + r" \\")

    lines.extend(
        [
            r"\botrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )
    return "\n".join(lines)


latex_table = generate_latex_table(formatted_table)
print(latex_table)

OUTPUT_TEX.write_text(latex_table + "\n", encoding="utf-8")
print(f"\nSaved LaTeX table to: {OUTPUT_TEX}")

% TABLE 1: Basic statistics:
\begin{table}[h]
\caption{Classification results across cross-validation subsets.}\label{tab:qnn_integration_results}%
\begin{tabular}{@{}lllllll@{}}
\toprule
Model name & Encoding / preprocessing & Accuracy & Precision & Recall & F1 & Loss \\
\midrule
Baseline A: &  & 0.845 $\pm$ 0.018 & 0.849 $\pm$ 0.018 & 0.845 $\pm$ 0.018 & 0.844 $\pm$ 0.018 & 0.792 $\pm$ 0.048 \\
Baseline B: &  & 0.852 $\pm$ 0.018 & 0.862 $\pm$ 0.014 & 0.852 $\pm$ 0.018 & 0.851 $\pm$ 0.018 & 0.487 $\pm$ 0.080 \\
Baseline C: &  & 0.121 $\pm$ 0.034 & 0.131 $\pm$ 0.024 & 0.121 $\pm$ 0.034 & 0.099 $\pm$ 0.031 & 17.352 $\pm$ 0.905 \\
Baseline D: &  & 0.503 $\pm$ 0.053 & 0.560 $\pm$ 0.033 & 0.503 $\pm$ 0.053 & 0.485 $\pm$ 0.059 & 10.141 $\pm$ 1.828 \\
GEQIE-FRQI: &  & 0.763 $\pm$ 0.038 & 0.787 $\pm$ 0.032 & 0.763 $\pm$ 0.038 & 0.757 $\pm$ 0.040 & 0.796 $\pm$ 0.115 \\
GEQIE-NEQR: &  & 0.766 $\pm$ 0.025 & 0.785 $\pm$ 0.017 & 0.766 $\pm$ 0.025 & 0.762 $\pm$ 0.025 & 0.818 $\pm$ 0.081 \\
\botrule